# Distributions in Depth

Notebook for **Chapter 2** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 Discrete Distributions: Counting Successes and Events

In [ ]:
from scipy import stats
b = stats.binom(20, 0.03)
print("P(X=2)  =", round(b.pmf(2), 4))
print("P(X<=1) =", round(b.cdf(1), 4))
print("P(X>=3) =", round(b.sf(2), 4), "  (sf = survival function = 1 - cdf)")
print("check sum:", round(b.cdf(1) + b.pmf(2) + b.sf(2), 6))

P(X=2)  = 0.0988
P(X<=1) = 0.8802
P(X>=3) = 0.021   (sf = survival function = 1 - cdf)
check sum: 1.0


In [ ]:
import numpy as np
p = stats.poisson(4)
print("P(0)   =", round(p.pmf(0), 4))
print("P(>8)  =", round(p.sf(8), 4))
print("99th percentile:", p.ppf(0.99), "  P(>9) =", round(p.sf(9), 4))
sim = np.random.default_rng(10).poisson(4, 1_000_000)
print("simulated P(>8):", round((sim > 8).mean(), 4), " P(>9):", round((sim > 9).mean(), 4))

P(0)   = 0.0183
P(>8)  = 0.0214
99th percentile: 9.0   P(>9) = 0.0081
simulated P(>8): 0.0214  P(>9): 0.0082


In [ ]:
rng = np.random.default_rng(11)
bin_pmf, poi_pmf = stats.binom(10000, 0.0005).pmf(8), stats.poisson(5).pmf(8)
print("binom pmf(8):", round(bin_pmf, 5), " poisson pmf(8):", round(poi_pmf, 5))
xb = rng.binomial(10000, 0.0005, 200_000); xp = rng.poisson(5, 200_000)
print("binom   mean/var:", round(xb.mean(), 3), round(xb.var(), 3))
print("poisson mean/var:", round(xp.mean(), 3), round(xp.var(), 3))

binom pmf(8): 0.06527  poisson pmf(8): 0.06528
binom   mean/var: 5.005 4.988
poisson mean/var: 5.0 4.981


## 2.2 Continuous Distributions for Time, Size and Noise

In [ ]:
import numpy as np
from scipy import stats
ex = stats.expon(scale=0.25)           # scale = mean = 1/rate
print("P(gap>1)       :", round(ex.sf(1), 4))
print("median gap     :", round(ex.median(), 4))
print("P(T>2 | T>1)   :", round(ex.sf(2) / ex.sf(1), 4))
gaps = np.random.default_rng(12).exponential(0.25, 1_000_000)
print("sim mean/median:", round(gaps.mean(), 4), round(np.median(gaps), 4))

P(gap>1)       : 0.0183
median gap     : 0.1733
P(T>2 | T>1)   : 0.0183
sim mean/median: 0.25 0.1734


In [ ]:
rng = np.random.default_rng(13)
data = rng.lognormal(5.3, 0.5, 2000)           # pretend these are measured delivery times (minutes)
mu, sd = np.log(data).mean(), np.log(data).std()
m, s = data.mean(), data.std()
ll_log = stats.lognorm(s=sd, scale=np.exp(mu)).logpdf(data).sum()
ll_nor = stats.norm(m, s).logpdf(data).sum()
print(f"lognormal MLE mu={mu:.3f} sigma={sd:.3f}   loglik={ll_log:.1f}")
print(f"normal    MLE mean={m:.1f} sd={s:.1f}  loglik={ll_nor:.1f}")
print("empirical 99th percentile:", round(np.percentile(data, 99), 1))
print("lognormal p99:", round(stats.lognorm(s=sd, scale=np.exp(mu)).ppf(0.99), 1))
print("normal    p99:", round(stats.norm(m, s).ppf(0.99), 1))
print("normal P(time < 0):", round(stats.norm(m, s).cdf(0), 4))

lognormal MLE mu=5.300 sigma=0.498   loglik=-12045.0
normal    MLE mean=227.2 sd=124.2  loglik=-12481.3
empirical 99th percentile: 670.5
lognormal p99: 638.6
normal    p99: 516.1
normal P(time < 0): 0.0337


In [ ]:
w = stats.weibull_min(c=1.5, scale=4)
print("S(3) =", round(w.sf(3), 4), " median =", round(w.median(), 3))
h = lambda t: w.pdf(t) / w.sf(t)
print("hazard at 1 yr:", round(h(1), 4), " at 5 yr:", round(h(5), 4))

S(3) = 0.5223  median = 3.133
hazard at 1 yr: 0.1875  at 5 yr: 0.4193


In [ ]:
# plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 3, figsize=(10, 5.2))
x = np.linspace(0, 12, 400)
ax[0,0].bar(range(0, 13), stats.poisson(4).pmf(range(0, 13)), color="#1f77b4"); ax[0,0].set_title("Poisson(4)")
ax[0,1].plot(x, stats.expon(scale=0.25).pdf(x), color="#d62728"); ax[0,1].set_title("Exponential (mean 0.25 min)"); ax[0,1].set_xlim(0, 2)
xs = np.linspace(0.01, 1200, 600)
ax[0,2].plot(xs, stats.lognorm(0.5, scale=200).pdf(xs), label="log-normal"); ax[0,2].plot(xs, stats.norm(227, 119).pdf(xs), ls="--", label="normal fit"); ax[0,2].legend(fontsize=7); ax[0,2].set_title("Delivery time (minutes)")
xt = np.linspace(0, 10, 300)
for k in (0.7, 1.0, 1.5, 3.0): ax[1,0].plot(xt, stats.weibull_min(k, scale=4).pdf(xt), label=f"k={k}")
ax[1,0].legend(fontsize=7); ax[1,0].set_title("Weibull (scale 4)")
xb = np.linspace(0, 1, 300)
for a_, b_ in [(1,1),(2,5),(8,2),(0.5,0.5)]: ax[1,1].plot(xb, stats.beta(a_, b_).pdf(xb), label=f"Beta({a_},{b_})")
ax[1,1].legend(fontsize=7); ax[1,1].set_ylim(0, 4); ax[1,1].set_title("Beta")
xg = np.linspace(0, 20, 300)
for k_ in (1, 2, 5, 10): ax[1,2].plot(xg, stats.chi2(k_).pdf(xg), label=f"df={k_}")
ax[1,2].legend(fontsize=7); ax[1,2].set_ylim(0, 0.5); ax[1,2].set_title("Chi-square")
plt.tight_layout(); plt.savefig(f"{IMG}/ch02_distributions.png", dpi=120); plt.close()

## 2.3 The Distributions That Come From the Normal: Chi-square, t and F

In [ ]:
rng = np.random.default_rng(14)
N, k = 500_000, 5
Z = rng.standard_normal(N)
V = (rng.standard_normal((N, k)) ** 2).sum(axis=1)
T = Z / np.sqrt(V / k)
print("sim  97.5% :", round(np.percentile(T, 97.5), 3), " P(T>2):", round((T > 2).mean(), 4))
print("t(5) exact :", round(stats.t(5).ppf(0.975), 3), " P(T>2):", round(stats.t(5).sf(2), 4))
print("normal     :", round(stats.norm.ppf(0.975), 3), " P(Z>2):", round(stats.norm.sf(2), 4))

sim  97.5% : 2.564  P(T>2): 0.0505
t(5) exact : 2.571  P(T>2): 0.051
normal     : 1.96  P(Z>2): 0.0228


In [ ]:
print("z 97.5%:", round(stats.norm.ppf(0.975), 3))
for df in (9, 29, 1000):
    print(f"t({df}) 97.5%:", round(stats.t(df).ppf(0.975), 3))
print("chi2(1) 95%:", round(stats.chi2(1).ppf(0.95), 3), " chi2(4) 95%:", round(stats.chi2(4).ppf(0.95), 3))
print("F(3,20) 95%:", round(stats.f(3, 20).ppf(0.95), 3), "  1.96^2 =", round(1.96**2, 3))

z 97.5%: 1.96
t(9) 97.5%: 2.262
t(29) 97.5%: 2.045
t(1000) 97.5%: 1.962
chi2(1) 95%: 3.841  chi2(4) 95%: 9.488
F(3,20) 95%: 3.098   1.96^2 = 3.842


## 2.4 Beta and Dirichlet: Distributions Over Probabilities

In [ ]:
for a_, b_ in [(4, 8), (301, 701)]:
    d = stats.beta(a_, b_)
    lo, hi = d.ppf([0.025, 0.975])
    print(f"Beta({a_},{b_}): mean={d.mean():.4f} 95% interval=({lo:.3f}, {hi:.3f}) width={hi-lo:.3f} P(>0.5)={d.sf(0.5):.4f}")

Beta(4,8): mean=0.3333 95% interval=(0.109, 0.610) width=0.500 P(>0.5)=0.1133
Beta(301,701): mean=0.3004 95% interval=(0.272, 0.329) width=0.057 P(>0.5)=0.0000


In [ ]:
alpha = np.array([51, 31, 16, 6])
print("mean shares:", np.round(alpha / alpha.sum(), 3), " sum:", round((alpha / alpha.sum()).sum(), 3))
draws = np.random.default_rng(15).dirichlet(alpha, 200_000)
print("P(button 1 highest):", round((draws.argmax(axis=1) == 0).mean(), 4))

mean shares: [0.49  0.298 0.154 0.058]  sum: 1.0
P(button 1 highest): 0.9876


## 2.5 Joint, Marginal and Conditional Distributions; the Multivariate Normal; Transforming Variables

In [ ]:
mean = [50, 200]
cov = [[100, 0.8 * 10 * 40], [0.8 * 10 * 40, 1600]]
xy = np.random.default_rng(16).multivariate_normal(mean, cov, 2_000_000)
sl = xy[(xy[:, 0] > 69) & (xy[:, 0] < 71)]
print("cov matrix:", cov)
print("slice size:", len(sl), " mean delivery time:", round(sl[:, 1].mean(), 1), " sd:", round(sl[:, 1].std(), 1))
print("theory: 264.0 and 24.0;  corr:", round(np.corrcoef(xy.T)[0, 1], 3))

cov matrix: [[100, 320.0], [320.0, 1600]]
slice size: 21555  mean delivery time: 263.7  sd: 24.1
theory: 264.0 and 24.0;  corr: 0.8


In [ ]:
x = np.random.default_rng(17).standard_normal(2_000_000)
print("E[e^X] simulated:", round(np.exp(x).mean(), 4), " theory:", round(np.exp(0.5), 4))
t, p, n = 0.5, 0.2, 3
mgf_product = (1 - p + p * np.exp(t)) ** n
mgf_binom = sum(stats.binom(n, p).pmf(k) * np.exp(t * k) for k in range(n + 1))
print("MGF product:", round(mgf_product, 6), " binomial MGF:", round(mgf_binom, 6))

E[e^X] simulated: 1.6508  theory: 1.6487
MGF product: 1.441918  binomial MGF: 1.441918


## 2.6 Law of Large Numbers and Markov Chains

In [ ]:
P = np.array([[0.6, 0.3, 0.1], [0.2, 0.5, 0.3], [0.5, 0.0, 0.5]])
w, v = np.linalg.eig(P.T)
pi = np.real(v[:, np.argmin(abs(w - 1))]); pi = pi / pi.sum()
print("stationary (eigen):", np.round(pi, 4))
x = np.array([1.0, 0.0, 0.0])
for _ in range(100): x = x @ P
print("after 100 steps   :", np.round(x, 4))
print("max |pi P - pi|   :", np.abs(pi @ P - pi).max().round(12))

stationary (eigen): [0.463  0.2778 0.2593]
after 100 steps   : [0.463  0.2778 0.2593]
max |pi P - pi|   : 0.0


In [ ]:
rng = np.random.default_rng(18)
norm = rng.normal(10, 2, 100_000); cau = rng.standard_cauchy(100_000)
for n in (100, 10_000, 100_000):
    print(f"n={n:<7} normal mean={norm[:n].mean():7.3f}   cauchy mean={cau[:n].mean():9.3f}")

n=100     normal mean=  9.962   cauchy mean=   -0.503
n=10000   normal mean=  9.988   cauchy mean=    0.326
n=100000  normal mean= 10.002   cauchy mean=   -0.108


## Chapter recap

## Practice set

Problem 2.7.1 X is Binomial(10, 0.5). What is P(X = 5)?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch02.html